# 03 · Gold layer: model features and claim-rate summary

**Project question.** Which drivers are likely to file a car insurance claim (`OUTCOME = 1`), and which characteristics are associated with that risk?

**Purpose.** Build the two analytics-ready tables that the modelling notebook and the report use.

**Inputs**

- `workspace.car_insurance.claims_silver` (from `02_silver_layer`)
- `workspace.car_insurance.claims_split_bronze` (from `01_bronze_layer`)

**Outputs**

| Table | Purpose | Intended use |
|---|---|---|
| `claims_gold_features` | One row per driver: target, model inputs, and the Phase 2 split label | Input to every Spark MLlib model in `04_mllib_models` |
| `claims_gold_summary` | Claim rate for each category of each driver characteristic | Report tables and figures; a first answer to the project question for a non-technical reader |

**Assumptions**

- Notebooks 01 and 02 have been run.
- Runs on serverless compute and can be rerun from top to bottom.

In [ ]:
# --- Configuration (identical in every Phase 3 notebook) ----------------------
# Free Edition workspaces come with a Unity Catalog catalog named `workspace`.
# If yours is named differently, change CATALOG here and in the other notebooks.
CATALOG = "workspace"
SCHEMA = "car_insurance"
VOLUME = "raw"

RAW_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CLAIMS_CSV = f"{RAW_DIR}/Car_Insurance_Claim.csv"
SPLIT_CSV = f"{RAW_DIR}/split_assignment.csv"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.claims_bronze"
SPLIT_BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.claims_split_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.claims_silver"
GOLD_FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_features"
GOLD_SUMMARY_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_summary"

# The 19 columns of the source file, in file order.
SOURCE_COLUMNS = [
    "ID", "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME",
    "CREDIT_SCORE", "VEHICLE_OWNERSHIP", "VEHICLE_YEAR", "MARRIED", "CHILDREN",
    "POSTAL_CODE", "ANNUAL_MILEAGE", "VEHICLE_TYPE", "SPEEDING_VIOLATIONS",
    "DUIS", "PAST_ACCIDENTS", "OUTCOME",
]

In [ ]:
from pyspark.sql import functions as F


def write_delta(df, table_name, comment):
    """Overwrite a managed Delta table so the notebook can be rerun top to bottom."""
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))
    spark.sql(f"COMMENT ON TABLE {table_name} IS '{comment}'")


def validate_table(table_name, expected_rows, expected_columns):
    """Read a table back from the catalog and compare it with what was written."""
    saved = spark.table(table_name)
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").first()
    rows, columns = saved.count(), len(saved.columns)
    passed = (detail["format"] == "delta" and rows == expected_rows
              and columns == expected_columns)
    return (table_name, detail["format"], expected_rows, rows,
            expected_columns, columns, "PASS" if passed else "FAIL")


VALIDATION_SCHEMA = ("table string, format string, expected_rows long, actual_rows long, "
                     "expected_columns int, actual_columns int, status string")


def show_validation(results):
    display(spark.createDataFrame(results, VALIDATION_SCHEMA))
    failed = [r[0] for r in results if r[-1] != "PASS"]
    assert not failed, f"Validation failed for: {failed}"

In [ ]:
silver = spark.table(SILVER_TABLE)
split_lookup = spark.table(SPLIT_BRONZE_TABLE)
silver_rows = silver.count()
print(f"Silver rows: {silver_rows:,}")

## 1. Model features: `claims_gold_features`

**Target.** `OUTCOME` (1 = the driver filed a claim).

**Features added here.** Only features that can be computed from a single driver's own row, so they are identical whether or not a driver is in the test set:

| Feature | Definition | Origin |
|---|---|---|
| `AGE` (relabelled) | `16-25` → Young-Adult, `26-39` → Adult, `40-64` → Middle Age, `65+` → Senior | Phase 1, Task 2 |
| `DRIVING_EXPERIENCE` (relabelled) | `0-9y` → Beginner, `10-19y` → Advanced, `20-29y` and `30y+` → Expert | Phase 1, Task 2 |
| `FAMILY` | 1 if the driver is married or has children, else 0 | Phase 1, Task 7 |

**Features left to the modelling notebook.** `INSURANCE_RISK` and `CAR_AGE` from Phase 1 depend on statistics of the data (min/max scaling and a median mileage), as do imputation, scaling and one-hot encoding. To avoid leakage they are fitted on the training split inside the Spark ML pipeline, so they are not stored in Gold.

**Split.** The `split` column is joined from the Phase 2 lookup so every model uses the same drivers as Phase 2: 6,400 train, 1,600 validation, 2,000 test.

In [ ]:
AGE_LABELS = {"16-25": "Young-Adult", "26-39": "Adult", "40-64": "Middle Age", "65+": "Senior"}
EXPERIENCE_LABELS = {"0-9y": "Beginner", "10-19y": "Advanced", "20-29y": "Expert", "30y+": "Expert"}


def relabel(column, mapping):
    """Map each known label to its new name; an unknown label becomes NULL and is caught below."""
    expression = None
    for old, new in mapping.items():
        condition = F.col(column) == old
        expression = F.when(condition, new) if expression is None else expression.when(condition, new)
    return expression.alias(column)


split_typed = split_lookup.select(F.expr("try_cast(ID AS bigint)").alias("ID"), "split")

FEATURE_COLUMNS = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME",
    "CREDIT_SCORE", "VEHICLE_OWNERSHIP", "VEHICLE_YEAR", "MARRIED", "CHILDREN",
    "FAMILY", "POSTAL_CODE", "ANNUAL_MILEAGE", "VEHICLE_TYPE",
    "SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS",
]

features_df = (
    silver
    .withColumn("AGE", relabel("AGE", AGE_LABELS))
    .withColumn("DRIVING_EXPERIENCE", relabel("DRIVING_EXPERIENCE", EXPERIENCE_LABELS))
    .withColumn("FAMILY", ((F.col("MARRIED") == 1) | (F.col("CHILDREN") == 1)).cast("int"))
    .join(split_typed, on="ID", how="left")   # left join: a driver without a split must not vanish
    .select("ID", "split", "OUTCOME", *FEATURE_COLUMNS)
)

write_delta(
    features_df,
    GOLD_FEATURES_TABLE,
    "Model-ready drivers: ID (key), split (Phase 2 assignment), OUTCOME (target) and 18 input features.",
)
print("Wrote", GOLD_FEATURES_TABLE)

In [ ]:
features = spark.table(GOLD_FEATURES_TABLE)
display(features.limit(10))

### Checks on the feature table

The split sizes and claim rates below should match Phase 2: 8,000 training drivers (6,400 train + 1,600 validation) at a 31.0% claim rate, and 2,000 test drivers at 32.8% with 656 claims.

In [ ]:
split_summary = (
    features.groupBy("split")
    .agg(F.count("*").alias("drivers"),
         F.sum("OUTCOME").alias("claims"),
         F.round(F.avg("OUTCOME"), 4).alias("claim_rate"))
    .orderBy("split")
)
display(split_summary)

split_counts = {row["split"]: (row["drivers"], row["claims"]) for row in split_summary.collect()}
feature_rows = features.count()

assert feature_rows == silver_rows, "The join changed the number of drivers"
assert feature_rows == features.select("ID").distinct().count(), "ID is repeated"
assert set(split_counts) == {"train", "validation", "test"}, f"Unexpected split labels: {set(split_counts)}"
assert features.filter(F.col("AGE").isNull() | F.col("DRIVING_EXPERIENCE").isNull()).count() == 0, \
    "A label was not covered by the relabelling"
assert split_counts["train"][0] == 6400 and split_counts["validation"][0] == 1600, "Split does not match Phase 2"
assert split_counts["test"] == (2000, 656), "Test split does not match Phase 2"
print("Split matches Phase 2: 6,400 train / 1,600 validation / 2,000 test (656 test claims).")

### Leakage check

| Column | Role | Known when a prediction is made? |
|---|---|---|
| `ID` | Row key. **Not a model input**: it is an arbitrary identifier | n/a |
| `split` | Phase 2 partition label. **Not a model input** | n/a |
| `OUTCOME` | Target | No, it is what is being predicted |
| 18 feature columns | Model inputs | Yes: driver demographics, vehicle details and driving history are all known when a policy is priced |

No feature is derived from `OUTCOME`, and nothing in this table is fitted on the data (no imputation, scaling or encoding), so test drivers cannot influence training through Gold.

The cells below look for features that track the target suspiciously closely, which would suggest a hidden copy of the outcome.

In [ ]:
NUMERIC_FEATURES = ["CREDIT_SCORE", "VEHICLE_OWNERSHIP", "MARRIED", "CHILDREN", "FAMILY",
                    "ANNUAL_MILEAGE", "SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]

correlations = features.select(
    [F.corr(name, "OUTCOME").alias(name) for name in NUMERIC_FEATURES]
).first().asDict()

display(
    spark.createDataFrame(
        sorted(((name, round(value, 4)) for name, value in correlations.items()),
               key=lambda item: -abs(item[1])),
        "numeric_feature string, correlation_with_outcome double",
    )
)
assert max(abs(v) for v in correlations.values()) < 0.9, "A feature is almost a copy of the target"

## 2. Claim-rate summary: `claims_gold_summary`

For each driver characteristic, this table gives the number of drivers, the number of claims and the claim rate per category, plus the **lift**: the category's claim rate divided by the overall claim rate. A lift of 2 means drivers in that category claim twice as often as average.

The table describes all 10,000 drivers and is meant for the report. It is built from the target, so it must **not** be joined back into the model features.

In [ ]:
from functools import reduce

from pyspark.sql import DataFrame

SUMMARY_DIMENSIONS = ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME",
                      "VEHICLE_OWNERSHIP", "VEHICLE_YEAR", "VEHICLE_TYPE", "MARRIED",
                      "CHILDREN", "FAMILY", "POSTAL_CODE"]

overall = features.agg(F.count("*").alias("drivers"), F.sum("OUTCOME").alias("claims")).first()
total_drivers, overall_rate = overall["drivers"], overall["claims"] / overall["drivers"]
print(f"Overall claim rate: {overall_rate:.4f} ({overall['claims']:,} of {total_drivers:,} drivers)")


def claim_rate_by(dimension):
    return (
        features.groupBy(F.col(dimension).cast("string").alias("category"))
        .agg(F.count("*").alias("drivers"), F.sum("OUTCOME").alias("claims"))
        .select(
            F.lit(dimension).alias("dimension"),
            "category",
            "drivers",
            "claims",
            F.round(F.col("drivers") / F.lit(total_drivers), 4).alias("share_of_drivers"),
            F.round(F.col("claims") / F.col("drivers"), 4).alias("claim_rate"),
            F.round(F.col("claims") / F.col("drivers") / F.lit(overall_rate), 2).alias("lift"),
        )
    )


summary_df = reduce(DataFrame.unionByName, [claim_rate_by(d) for d in SUMMARY_DIMENSIONS])

write_delta(
    summary_df,
    GOLD_SUMMARY_TABLE,
    "Claim rate and lift per category of each driver characteristic, over all drivers. For reporting only.",
)
print("Wrote", GOLD_SUMMARY_TABLE)

In [ ]:
summary = spark.table(GOLD_SUMMARY_TABLE)
summary_rows = summary.count()

# Every dimension must account for all drivers exactly once.
per_dimension = summary.groupBy("dimension").agg(F.sum("drivers").alias("drivers")).collect()
assert all(row["drivers"] == total_drivers for row in per_dimension), "A dimension lost or duplicated drivers"
assert len(per_dimension) == len(SUMMARY_DIMENSIONS)

display(summary.orderBy(F.desc("lift")))

In [ ]:
# Categories where the outcome is the same for every driver. A category like this
# lets a model memorise the target, so it is reported here as a caution.
display(
    summary.filter(F.col("claim_rate").isin(0.0, 1.0)).orderBy(F.desc("drivers"))
)

**Reading the summary.** The highest-lift categories are the first place to look when answering the project question, and they can be compared with the logistic regression odds ratios from Phase 2.

**Caution on `POSTAL_CODE = 21217`.** Every driver in this postal code filed a claim. This is the same artifact noted in the Phase 2 limitations: it is a property of the dataset, not of information leaking from the target, but any model that uses `POSTAL_CODE` will treat this area as a near-certain claim.

## 3. Delta validation: all pipeline tables

Final read-back of every table the pipeline produced, from the catalog. The format, row count and column count are compared with the documented values: 10,000 drivers in every row-level table.

In [ ]:
show_validation([
    validate_table(BRONZE_TABLE, 10000, len(SOURCE_COLUMNS) + 3),
    validate_table(SPLIT_BRONZE_TABLE, 10000, 4),
    validate_table(SILVER_TABLE, 10000, len(SOURCE_COLUMNS)),
    validate_table(GOLD_FEATURES_TABLE, 10000, len(FEATURE_COLUMNS) + 3),
    validate_table(GOLD_SUMMARY_TABLE, summary_rows, 7),
])

In [ ]:
features.printSchema()
summary.printSchema()

In [ ]:
# Delta keeps a transaction log: each rerun of the pipeline adds a new table version.
display(
    spark.sql(f"DESCRIBE HISTORY {GOLD_FEATURES_TABLE}")
    .select("version", "timestamp", "operation", "operationMetrics")
)

**Result.** The Gold layer provides `claims_gold_features` (10,000 drivers, target, 18 inputs, Phase 2 split) for modelling and `claims_gold_summary` for reporting. All five pipeline tables were read back as Delta with the expected row and column counts.

**How the modelling notebook should use the features table**

```python
features = spark.table("workspace.car_insurance.claims_gold_features")
train = features.filter("split = 'train'")
validation = features.filter("split = 'validation'")
test = features.filter("split = 'test'")
```

Phase 2 fitted its preprocessing on the 8,000 drivers in `train` + `validation`, so fit pipeline stages on those rows and keep `test` for the final evaluation only.

Next: `04_mllib_models`.